# Exploration de la couche Gold

Ces graphiques explorent les agrégats produits par `work/03_gold.ipynb`. Exécuter d'abord Silver, puis Gold. Les volumes mesurés et les moyennes par heure ne représentent ni le nombre de personnes ni la capacité des voies.

## 1. Connexion et lecture des tables Gold

Les tables sont lues dans le format actif du projet (Delta ou Parquet). Les agrégats sont de petite taille et peuvent être convertis en Pandas pour les graphiques.

In [ ]:
import sys
from pathlib import Path

racine_projet = Path.cwd()
if not (racine_projet / "commun").is_dir():
    racine_projet = racine_projet.parent
if not (racine_projet / "commun").is_dir():
    raise FileNotFoundError("Exécutez ce notebook depuis le dossier work du projet.")
sys.path.insert(0, str(racine_projet))

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import functions as F
from commun.spark_utils import LAKE, creer_session, format_tables

spark = creer_session("S04_ExplorationGold", lac=True, delta=True, postgres=False)
spark.conf.set("spark.sql.session.timeZone", "Europe/Paris")
FMT = format_tables(spark)
GOLD_PATH = f"{LAKE}/gold"

serie = spark.read.format(FMT).load(f"{GOLD_PATH}/gold_serie_mensuelle")
classement = spark.read.format(FMT).load(f"{GOLD_PATH}/gold_sites_classement")
heatmap = spark.read.format(FMT).load(f"{GOLD_PATH}/gold_heatmap_heure_jour")
kpi = spark.read.format(FMT).load(f"{GOLD_PATH}/gold_kpi")

for nom, table in [("gold_serie_mensuelle", serie), ("gold_sites_classement", classement),
                   ("gold_heatmap_heure_jour", heatmap), ("gold_kpi", kpi)]:
    print(f"{nom}: {table.count():,} lignes")
kpi.orderBy("kpi_id").show(truncate=False)

## 2. Évolution mensuelle et couverture

La moyenne des comptages par site-mode-heure permet de comparer les heures mesurées, tandis que le nombre de sites actifs aide à repérer une variation de couverture. Les deux mesures sont montrées séparément car elles n'ont pas la même unité.

In [ ]:
mensuel = serie.orderBy("annee", "mois", "mode_deplacement").toPandas()
if mensuel.empty:
    raise ValueError("La table gold_serie_mensuelle ne contient aucune ligne à tracer.")
mensuel["mois_date"] = pd.to_datetime(dict(year=mensuel["annee"], month=mensuel["mois"], day=1))

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for mode, groupe in mensuel.groupby("mode_deplacement"):
    groupe = groupe.sort_values("mois_date")
    axes[0].plot(groupe["mois_date"], groupe["moyenne_par_heure"], marker="o", markersize=3, label=mode)
    axes[1].plot(groupe["mois_date"], groupe["nb_sites_actifs"], marker="o", markersize=3, label=mode)
axes[0].set(title="Moyenne des comptages par site-mode-heure", ylabel="Véhicules par heure mesurée")
axes[1].set(title="Couverture mensuelle", xlabel="Mois", ylabel="Sites actifs")
for ax in axes:
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", alpha=0.25)
axes[0].legend(title="Mode", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

## 3. Avant/après : classement naïf et classement corrigé

Un classement naïf par total favorise les sites mesurés longtemps. Le classement corrigé utilise la moyenne par heure mesurée et affiche le nombre d'heures pour garder la durée de collecte visible. Les deux histogrammes commencent à zéro, avec une unité et une échelle propres à chaque panneau : leurs hauteurs ne se comparent pas directement; ce sont les rangs et les durées affichées qu'il faut examiner. La moyenne ne corrige toutefois ni la couverture ni la capacité des voies.

In [ ]:
naif = (classement
    .orderBy(F.desc("total_usagers"), F.desc("nb_heures_mesurees"), F.asc("identifiant_site"))
    .limit(15)
    .toPandas())
corrige = (classement
    .filter(F.col("moyenne_par_heure").isNotNull())
    .orderBy(F.desc("moyenne_par_heure"), F.desc("nb_heures_mesurees"), F.asc("identifiant_site"))
    .limit(15)
    .toPandas())
if naif.empty or corrige.empty:
    raise ValueError("La table gold_sites_classement ne contient aucun site à classer.")

def ajouter_libelle(table):
    table = table.copy()
    table["site"] = table["nom_du_site"].fillna(table["identifiant_site"]).astype(str)
    table["libelle"] = (table["site"] + " — " + table["mode_deplacement"].astype(str)
                        + " — " + table["annee"].astype(str) + " — "
                        + table["type_voie"].fillna("voie inconnue").astype(str)
                        + " (" + table["nb_heures_mesurees"].astype(str) + " h)")
    return table

naif = ajouter_libelle(naif).sort_values("total_usagers")
corrige = ajouter_libelle(corrige).sort_values("moyenne_par_heure")
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].barh(naif["libelle"], naif["total_usagers"], color="#a6a6a6")
axes[0].set(title="Naïf — classement par comptage cumulé", xlabel="Véhicules comptés")
axes[1].barh(corrige["libelle"], corrige["moyenne_par_heure"], color="#3973ac")
axes[1].set(title="Corrigé — classement par heure mesurée", xlabel="Véhicules par heure mesurée")
for ax in axes:
    ax.set_xlim(left=0)
    ax.grid(axis="x", alpha=0.25)
fig.suptitle("Effet de la durée de mesure sur le classement des sites")
fig.tight_layout()
plt.show()

comparaison = pd.concat([
    naif.assign(type_classement="Naïf — total"),
    corrige.assign(type_classement="Corrigé — moyenne par heure"),
], ignore_index=True)
comparaison[["type_classement", "identifiant_site", "mode_deplacement", "annee",
             "total_usagers", "moyenne_par_heure", "nb_heures_mesurees"]]

## 4. Profil moyen jour × heure par mode

Chaque panneau montre la moyenne des totaux site-mode-heure observés pour un mode. Les jours suivent la convention Spark : 1 = dimanche et 7 = samedi. Les couleurs sont interprétées à l'intérieur d'un mode, pas comme une comparaison absolue entre modes.

In [ ]:
jours = {1: "Dim", 2: "Lun", 3: "Mar", 4: "Mer", 5: "Jeu", 6: "Ven", 7: "Sam"}
profil = heatmap.toPandas()
modes = sorted(profil["mode_deplacement"].dropna().unique())
if not modes:
    raise ValueError("La table gold_heatmap_heure_jour ne contient aucun mode à tracer.")

fig, axes = plt.subplots(len(modes), 1, figsize=(14, 3.2 * len(modes)), sharex=True, squeeze=False)
for index, mode in enumerate(modes):
    matrice = (profil[profil["mode_deplacement"] == mode]
        .pivot(index="jour_semaine", columns="heure", values="moyenne_par_heure")
        .reindex(index=range(1, 8), columns=range(24)))
    ax = axes[index, 0]
    image = ax.imshow(matrice, aspect="auto", origin="upper", cmap="YlOrRd", vmin=0)
    ax.set_yticks(range(7), [jours[i] for i in range(1, 8)])
    ax.set_ylabel(mode)
    fig.colorbar(image, ax=ax, label="Véhicules par site-heure")
axes[-1, 0].set_xticks(range(24))
axes[-1, 0].set_xlabel("Heure locale")
fig.suptitle("Moyenne des comptages observés par jour et heure, par mode")
fig.tight_layout()
plt.show()